# Fase 4.0 — Validación del yaw de MediaPipe contra BIWI

**Proyecto:** Sparkle V2 · **Protocolo:** `Protocolo_Reglas_Geometricas.md` · `Anexo_G_Validacion_con_Datasets_Publicos.md`

Valida la **afirmación A** de la descomposición del anexo G:

> *El yaw que devuelve MediaPipe FaceLandmarker es preciso en grados.*

**Ground truth:** BIWI Kinect Head Pose Database — 15.000 imágenes, 20 sujetos, pose medida por
**sensor de profundidad** (no anotada por humanos), rango ±75° de yaw.

---

## Criterios declarados ANTES de correr

| # | Métrica | Objetivo |
| :--- | :--- | :--- |
| **U4a** | MAE del yaw, restringido a `\|yaw_gt\| ≤ 45°` (rango plausible sentado frente a una pantalla) | Reportado. Referencia de literatura: 5-8° |
| **U4b** | **Acuerdo de la decisión `\|yaw\| > 20°`** entre MediaPipe y ground truth | **≥ 0,90** ← *este es el que importa* |
| **U4c** | MAE en la banda `\|yaw_gt\| ∈ [15°, 25°]` — donde el error da vuelta la decisión | Reportado |

**Por qué U4b y no el MAE.** La regla R1 no usa el valor del yaw: usa si supera 20°. Un error de 4°
a 60° de giro es irrelevante; un error de 4° a 21° da vuelta la decisión. **El MAE global es la
métrica cómoda, no la relevante.**

**Restricción declarada a priori:** el análisis primario excluye `|yaw_gt| > 45°`. BIWI incluye
poses extremas que no ocurren en un estudiante sentado frente a una notebook. El rango completo se
reporta como secundario. **Esto se fija ahora, no después de ver resultados.**

> ⚠️ Usar `output_facial_transformation_matrixes=True`. Es **la misma vía** por la que la extensión
> obtiene yaw/pitch (ficha técnica §3.1). Calcular la pose con `solvePnP` sobre los landmarks daría
> otro número y **no validaría lo que está desplegado**.

## 1. Entorno

> ⚠️ **No pinear `mediapipe==0.10.14`.** Esa versión exige `protobuf<5` y al bajarlo rompe el
> TensorFlow preinstalado de Colab (`ImportError: cannot import name 'runtime_version'`).
> Se instala la última versión, que ya convive con protobuf 5.x.
>
> **La celda de instalación reinicia el entorno sola.** Es esperado: cuando el runtime vuelva,
> seguí desde la celda 1.2 sin volver a correr la 1.1.

In [ ]:
# --- 1.1 INSTALACION (reinicia el entorno al terminar) ---
!pip -q install -U mediapipe opencv-python-headless

import google.protobuf, importlib
print("protobuf:", google.protobuf.__version__)

# protobuf >= 5.27 expone runtime_version; si no esta, TF de Colab va a fallar
try:
    importlib.import_module("google.protobuf.runtime_version")
    print("runtime_version presente -> compatible con el TF de Colab")
except ImportError:
    print("!! protobuf quedo viejo. Corriendo: pip install -U 'protobuf>=5.28'")
    !pip -q install -U "protobuf>=5.28"

print("\nReiniciando el entorno. Cuando vuelva, seguir desde la celda 1.2.")
import os; os.kill(os.getpid(), 9)

In [ ]:
# --- 1.2 IMPORTS (correr DESPUES del reinicio) ---
import mediapipe as mp, cv2, numpy as np, pandas as pd, os, glob, math
import google.protobuf
print("mediapipe:", mp.__version__, "| protobuf:", google.protobuf.__version__)

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from scipy.io import loadmat        # para AFLW2000-3D
print("tasks API OK")

In [ ]:
# Modelo de MediaPipe (mismo que usa la extension)
!wget -q -O face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print(os.path.getsize("face_landmarker.task"), "bytes")

## 2. Datos — tres fuentes, en orden de preferencia

Kaggle devolvió **403 en `GetDatasetMetadata`**: credencial rechazada. Causas habituales, en orden:
token regenerado (invalida el anterior), cuenta sin verificar por teléfono, o términos del dataset
sin aceptar en la web. **Se agregan dos rutas que no dependen de Kaggle.**

| Fuente | Ground truth | Auth | Nota |
| :--- | :--- | :--- | :--- |
| **A — BIWI vía Hugging Face** | **Medido por sensor de profundidad** | no | **Preferida.** Mismo dataset, sin Kaggle |
| **B — AFLW2000-3D** | **Derivado por ajuste 3DMM**, no medido | no | Fallback. Ver la advertencia de §2.bis |
| **C — BIWI vía Kaggle** | Medido por sensor | sí | Requiere token nuevo |

Correr la celda 2.1: prueba A, y si falla pasa a B automáticamente.

In [ ]:
# --- 2.1 Descarga con fallback automatico ---
import os, glob, zipfile, subprocess, sys
FUENTE = None

# ---------- A: BIWI via Hugging Face ----------
try:
    !pip -q install -U huggingface_hub
    from huggingface_hub import snapshot_download
    from huggingface_hub import list_repo_files
    REPO = "ETHZurich/biwi_kinect_head_pose"
    print("archivos en el repo HF:")
    for f in list_repo_files(REPO, repo_type="dataset")[:30]: print("   ", f)
    ruta = snapshot_download(repo_id=REPO, repo_type="dataset",
                             local_dir="/content/biwi", max_workers=8)
    print("HF descargado en:", ruta)
    for z in glob.glob("/content/biwi/**/*.zip", recursive=True) + \
             glob.glob("/content/biwi/**/*.tgz", recursive=True) + \
             glob.glob("/content/biwi/**/*.tar.gz", recursive=True):
        print("  extrayendo", os.path.basename(z))
        if z.endswith(".zip"):
            zipfile.ZipFile(z).extractall("/content/biwi")
        else:
            subprocess.run(["tar","-xzf",z,"-C","/content/biwi"], check=False)
    if glob.glob("/content/biwi/**/*_pose.txt", recursive=True):
        FUENTE = "BIWI"
    else:
        print("  HF bajo, pero no aparecen *_pose.txt. Revisar contenido:")
        print("  ", os.listdir("/content/biwi")[:20])
except Exception as e:
    print("A (Hugging Face) fallo:", type(e).__name__, e)

# ---------- B: AFLW2000-3D ----------
if FUENTE is None:
    print("\n--> Fallback a AFLW2000-3D")
    !wget -q --show-progress -O /content/aflw2000.zip \
      http://www.cbsr.ia.ac.cn/users/xiangyuzhu/projects/3DDFA/Database/AFLW2000-3D.zip
    try:
        zipfile.ZipFile("/content/aflw2000.zip").extractall("/content/aflw2000")
        if glob.glob("/content/aflw2000/**/*.mat", recursive=True):
            FUENTE = "AFLW2000"
    except Exception as e:
        print("B (AFLW2000) fallo:", type(e).__name__, e)

assert FUENTE, ("Ninguna fuente disponible. Opcion C: generar un token nuevo en "
                "kaggle.com/settings -> API -> Create New Token, aceptar las reglas del dataset "
                "en su pagina, y volver a intentar con la celda de Kaggle.")
print("\nFUENTE ACTIVA:", FUENTE)

### 2.bis ⚠️ Si la fuente activa es AFLW2000-3D, cambia lo que se puede afirmar

| | BIWI | AFLW2000-3D |
| :--- | :--- | :--- |
| Origen del ground truth | **Medición por sensor de profundidad** | **Ajuste de un modelo 3DMM** — es una estimación |
| Ruido de la etiqueta | ≈ 0 | **No despreciable, y no cuantificado** |
| Sujetos | 20, identificables | Imágenes sueltas *in the wild*, sin identidad |
| Bootstrap | **Clusterizado por sujeto** (regla 3 del protocolo) | **i.i.d. sobre imágenes** — más optimista |

**Con AFLW2000 el argumento del anexo G §4 se debilita:** la ventaja de "ground truth medido y no
anotado" desaparece. Sigue siendo un benchmark público y citable, pero **hay que declarar en la
tesis que el ground truth es derivado de un ajuste de modelo**, y que el IC es i.i.d. sobre
imágenes y no clusterizado.

**Recomendación: insistir con BIWI** (Hugging Face o token nuevo de Kaggle) antes de conformarse
con el fallback.

In [ ]:
# --- 2.2 Armado de la lista (imagen, pose) segun la fuente ---
import numpy as np, math, os, glob
pares = []

if FUENTE == "BIWI":
    for pz in glob.glob("/content/biwi/**/*_pose.txt", recursive=True):
        for ext in ("_rgb.png", "_rgb.jpg"):
            img = pz.replace("_pose.txt", ext)
            if os.path.exists(img):
                pares.append((img, pz, os.path.basename(os.path.dirname(img))))  # sujeto = carpeta
                break

elif FUENTE == "AFLW2000":
    for m in glob.glob("/content/aflw2000/**/*.mat", recursive=True):
        img = m[:-4] + ".jpg"
        if os.path.exists(img):
            pares.append((img, m, os.path.basename(img)))   # sin identidad: cada imagen es su cluster

print(f"fuente={FUENTE}  pares imagen+pose: {len(pares)}")
assert pares, "No se encontraron pares. Inspeccionar: !find /content -name '*_pose.txt' -o -name '*.mat' | head"
print(pares[0])

## 3. Convenciones de ángulo

BIWI entrega una **matriz de rotación 3×3** de la cabeza. MediaPipe entrega una **matriz de
transformación facial 4×4**. Hay que extraer Euler de las dos **con la misma convención**, y
**verificar el signo empíricamente** en vez de suponerlo.

In [ ]:
def euler_desde_R(R):
    """Devuelve (yaw, pitch, roll) en grados. Convencion fija para ambas fuentes."""
    sy = math.sqrt(R[0,0]**2 + R[1,0]**2)
    if sy > 1e-6:
        x = math.atan2( R[2,1], R[2,2])   # pitch
        y = math.atan2(-R[2,0], sy)       # yaw
        z = math.atan2( R[1,0], R[0,0])   # roll
    else:
        x = math.atan2(-R[1,2], R[1,1]); y = math.atan2(-R[2,0], sy); z = 0.0
    return math.degrees(y), math.degrees(x), math.degrees(z)

def leer_pose(path, fuente):
    """Devuelve (yaw, pitch, roll) en grados, sea BIWI (_pose.txt) o AFLW2000 (.mat)."""
    if fuente == "BIWI":
        v = np.loadtxt(path)
        R = v[:3, :3] if v.shape[0] >= 3 else np.array(v[:9]).reshape(3, 3)
        return euler_desde_R(R)
    else:                                   # AFLW2000-3D
        from scipy.io import loadmat
        # Pose_Para = [pitch, yaw, roll, tx, ty, tz, escala], en RADIANES
        pp = loadmat(path)["Pose_Para"][0]
        pitch, yaw, roll = (math.degrees(a) for a in pp[:3])
        return yaw, pitch, roll

## 4. Extracción con MediaPipe

In [ ]:

opts = vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="face_landmarker.task"),
    output_facial_transformation_matrixes=True,   # <-- la via que usa la extension
    num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(opts)

MAX = None          # poner un entero para una corrida rapida de prueba
rows, sin_rostro = [], 0
for k, (img_p, pose_p, sujeto) in enumerate(pares[:MAX] if MAX else pares):
    bgr = cv2.imread(img_p)
    if bgr is None: continue
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    res = landmarker.detect(mp_img)
    if not res.facial_transformation_matrixes:
        sin_rostro += 1; continue
    M = np.array(res.facial_transformation_matrixes[0]).reshape(4,4)
    yaw_mp, pitch_mp, roll_mp = euler_desde_R(M[:3,:3])
    yaw_gt, pitch_gt, roll_gt = leer_pose(pose_p, FUENTE)
    rows.append(dict(img=img_p, sujeto=sujeto,
                     yaw_mp=yaw_mp, yaw_gt=yaw_gt, pitch_mp=pitch_mp, pitch_gt=pitch_gt))
    if k % 500 == 0: print(k, "/", len(pares))

df = pd.DataFrame(rows)
print(f"\nprocesadas: {len(df)}   sin rostro detectado: {sin_rostro} "
      f"({100*sin_rostro/max(1,len(pares)):.2f} %)")
df.to_csv(f"validacion_yaw_{FUENTE}.csv", index=False)
df.head()

## 5. Verificación de signo

**Antes de medir el error hay que confirmar que las dos fuentes usan el mismo signo.** Si la
correlación diera fuertemente negativa, la convención está invertida: se registra y se corrige
**explícitamente**, nunca en silencio.

In [ ]:
r = np.corrcoef(df.yaw_mp, df.yaw_gt)[0,1]
print(f"correlacion yaw MediaPipe vs {FUENTE}: {r:+.4f}")
if r < -0.5:
    print(">>> SIGNO INVERTIDO. Se invierte yaw_mp y queda REGISTRADO.")
    df["yaw_mp"] = -df["yaw_mp"]
elif r < 0.5:
    print(">>> ATENCION: correlacion baja. Revisar la convencion antes de seguir.")
else:
    print(">>> Convencion coincidente.")

## 6. Resultados

In [ ]:
UMBRAL = 20.0     # congelado en Protocolo_Reglas_Geometricas.md 2.1
df["err"] = df.yaw_mp - df.yaw_gt

def reporte(d, nombre):
    if len(d) == 0: print(f"{nombre}: sin datos"); return
    mae  = d.err.abs().mean(); bias = d.err.mean()
    fuera_gt = d.yaw_gt.abs() > UMBRAL
    fuera_mp = d.yaw_mp.abs() > UMBRAL
    acc = (fuera_gt == fuera_mp).mean()
    fp = ((~fuera_gt) & fuera_mp).sum() / max(1, (~fuera_gt).sum())
    fn = (fuera_gt & (~fuera_mp)).sum() / max(1, fuera_gt.sum())
    print(f"\n--- {nombre}  (n={len(d)}) ---")
    print(f"  MAE          : {mae:6.2f}°     sesgo: {bias:+.2f}°")
    print(f"  p90 |error|  : {d.err.abs().quantile(.90):6.2f}°")
    print(f"  ACUERDO de la decision |yaw|>{UMBRAL:.0f}° : {acc:.4f}")
    print(f"    falsos positivos (GT dentro, MP fuera) : {fp:.4f}")
    print(f"    falsos negativos (GT fuera, MP dentro) : {fn:.4f}")
    return acc

print("="*62); print(f"FUENTE: {FUENTE}")
print("PRIMARIO — rango plausible sentado (declarado a priori)")
acc_primario = reporte(df[df.yaw_gt.abs() <= 45], "U4a/U4b  |yaw_gt| <= 45°")
print("\n" + "="*62); print("SECUNDARIOS")
reporte(df, f"rango completo de {FUENTE} (incluye poses extremas)")
reporte(df[df.yaw_gt.abs().between(15, 25)], "U4c  banda critica |yaw_gt| in [15,25]")

print("\n" + "="*62)
print(f"CRITERIO U4b: acuerdo >= 0,90  ->  "
      f"{'CUMPLE' if acc_primario and acc_primario >= 0.90 else 'NO CUMPLE'}"
      f"  (obtenido {acc_primario:.4f})")

## 6.bis Diagnóstico parcial del sesgo compartido

**El problema.** El ground truth de AFLW2000-3D sale de un **ajuste 3DMM sobre landmarks**;
MediaPipe estima la pose **también desde un modelo facial**. Comparten familia de método, así que
podrían compartir sesgos sistemáticos e inflar el acuerdo.

**Lo que este diagnóstico SÍ detecta.** El modo de falla más probable es un **error de escala**: que
MediaPipe comprima o expanda el rango angular. Se detecta con la regresión `yaw_mp = a + b·yaw_gt`:

- `b ≈ 1` → las dos fuentes coinciden en escala;
- `b < 1` → MediaPipe **encoge** el ángulo, y R1 dispararía **de menos** cerca del umbral;
- `b > 1` → lo exagera, y dispararía de más.

**⚠️ Lo que este diagnóstico NO puede detectar.** Si **ambas** fuentes tuvieran el mismo sesgo,
`b` daría 1 igual. **El sesgo de modo común solo se descarta con una medición de otra modalidad**
—un sensor de profundidad, es decir BIWI—. **Esa limitación se declara en la tesis y no se
disimula.**

In [ ]:
d = df[df.yaw_gt.abs() <= 45]
b, a = np.polyfit(d.yaw_gt, d.yaw_mp, 1)
r2 = np.corrcoef(d.yaw_gt, d.yaw_mp)[0,1]**2

rng = np.random.default_rng(42)
bs = [np.polyfit(*(lambda i: (d.yaw_gt.values[i], d.yaw_mp.values[i]))(
        rng.integers(0, len(d), len(d))), 1)[0] for _ in range(2000)]
lo, hi = np.percentile(bs, [2.5, 97.5])

print(f"yaw_mp = {a:+.3f} + {b:.4f} * yaw_gt     R² = {r2:.4f}   n = {len(d)}")
print(f"pendiente b = {b:.4f}   IC 95% [{lo:.4f} - {hi:.4f}]")
print()
if lo <= 1.0 <= hi:
    print("  -> b = 1 esta DENTRO del IC: no hay error de escala detectable.")
else:
    tipo = "ENCOGE" if b < 1 else "EXAGERA"
    print(f"  -> b = 1 QUEDA FUERA del IC: MediaPipe {tipo} el angulo de forma sistematica.")
    print(f"     A 20° de giro real, mide {20*b:+.1f}° -> R1 dispararia "
          f"{'de menos' if b<1 else 'de mas'} cerca del umbral.")
print()
print("  RECORDATORIO: esto NO descarta sesgo de modo comun (ambas fuentes equivocadas igual).")
print("  Eso requiere ground truth de otra modalidad (sensor de profundidad). Se declara como")
print("  limitacion abierta.")

In [ ]:
# IC bootstrap. Clusterizado POR SUJETO si la fuente los tiene (regla 3 del protocolo);
# i.i.d. sobre imagenes si no (AFLW2000), y se DECLARA.
rng = np.random.default_rng(42)
d = df[df.yaw_gt.abs() <= 45]
acierto = ((d.yaw_gt.abs() > UMBRAL) == (d.yaw_mp.abs() > UMBRAL)).values

if FUENTE == "BIWI":
    sujs = d.sujeto.unique()
    grupos = [acierto[(d.sujeto == s).values] for s in sujs]
    boot = [np.concatenate([grupos[i] for i in rng.integers(0, len(grupos), len(grupos))]).mean()
            for _ in range(2000)]
    print(f"bootstrap CLUSTERIZADO por sujeto  (n sujetos = {len(sujs)})")
else:
    boot = [acierto[rng.integers(0, len(acierto), len(acierto))].mean() for _ in range(2000)]
    print("bootstrap i.i.d. sobre imagenes (AFLW2000 no expone identidad de sujeto).")
    print("    Defendible aca: son mayormente caras distintas, asi que el cluster es la imagen.")
    print("    Lo que SI hay que declarar es que el ground truth es derivado de un ajuste 3DMM.")

boot = np.array(boot)
print(f"Acuerdo de decision: {np.median(boot):.4f}  "
      f"IC 95% [{np.percentile(boot,2.5):.4f} - {np.percentile(boot,97.5):.4f}]")

## 7. Figura

In [ ]:
import matplotlib.pyplot as plt
d = df[df.yaw_gt.abs() <= 45]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
ax[0].scatter(d.yaw_gt, d.yaw_mp, s=3, alpha=.15)
lim = [-46, 46]; ax[0].plot(lim, lim, 'k--', lw=1)
for v in (-UMBRAL, UMBRAL):
    ax[0].axvline(v, color='r', ls=':', lw=1); ax[0].axhline(v, color='r', ls=':', lw=1)
ax[0].set_xlabel(f"yaw ground truth {FUENTE} (°)"); ax[0].set_ylabel("yaw MediaPipe (°)")
ax[0].set_title(f"Yaw estimado vs. referencia ({FUENTE})\n(líneas rojas: umbral de 20° de R1)")
ax[0].set_xlim(lim); ax[0].set_ylim(lim); ax[0].grid(alpha=.25)
ax[1].hist(d.err, bins=80); ax[1].axvline(0, color='k', lw=1)
ax[1].set_xlabel(f"error (MediaPipe − {FUENTE}), grados"); ax[1].set_ylabel("frecuencia")
ax[1].set_title(f"Distribución del error · MAE {d.err.abs().mean():.2f}°"); ax[1].grid(alpha=.25)
plt.tight_layout(); plt.savefig(f"validacion_yaw_{FUENTE}.png", dpi=160); plt.show()

## 8. Qué hacer con el resultado

| Resultado de U4b | Lectura | Acción |
| :--- | :--- | :--- |
| **≥ 0,95** | El yaw es fiable en el umbral | La afirmación A queda demostrada. Seguir con CEW (R2) |
| **0,90 – 0,95** | Aceptable | Se reporta el acuerdo y el MAE de la banda crítica junto a toda métrica de R1 |
| **< 0,90** | El yaw no es fiable a 20° | **No se toca el umbral para que dé.** Se reporta el hallazgo y se evalúa subir el umbral **por enmienda fechada**, con las tres respuestas del protocolo §9 |

**Guardar en el repo:** `validacion_yaw_<FUENTE>.csv`, `validacion_yaw_biwi.png`, y una entrada en
`Fase4_Bitacora_de_Ejecucion.md` con los tres números (MAE primario, acuerdo con IC, MAE de la
banda crítica) y la tasa de no-detección de rostro.

> **Recordatorio de método (anexo G §8).** Esto valida **el detector**, no **la regla**. Que el yaw
> sea preciso no prueba que 20° durante el 30 % de la ventana sea buen criterio de desatención: eso
> es una decisión de diseño derivada de la geometría de la pantalla, y se declara como tal.